# Question 12 - 380. Insert Delete GetRandom O(1)

Implement the `RandomizedSet` class:

- `RandomizedSet()` Initializes the `RandomizedSet` object.
- `bool insert(int val)` Inserts an item `val` into the set if not present. Returns `true` if the item was not present, `false` otherwise.
- `bool remove(int val)` Removes an item `val` from the set if present. Returns `true` if the item was present, `false` otherwise.
- `int getRandom()` Returns a random element from the current set of elements (it's guaranteed that at least one element exists when this method is called). Each element must have the **same probability** of being returned.

You must implement the functions of the class such that each function works in **average** `O(1)` time complexity.

**Example 1:**

    Input
    ["RandomizedSet", "insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
    [[], [1], [2], [2], [], [1], [2], []]
    Output
    [null, true, false, true, 2, true, false, 2]

    Explanation
    RandomizedSet randomizedSet = new RandomizedSet();
    randomizedSet.insert(1); // Inserts 1 to the set. Returns true as 1 was inserted successfully.
    randomizedSet.remove(2); // Returns false as 2 does not exist in the set.
    randomizedSet.insert(2); // Inserts 2 to the set, returns true. Set now contains [1,2].
    randomizedSet.getRandom(); // getRandom() should return either 1 or 2 randomly.
    randomizedSet.remove(1); // Removes 1 from the set, returns true. Set now contains [2].
    randomizedSet.insert(2); // 2 was already in the set, so return false.
    randomizedSet.getRandom(); // Since 2 is the only number in the set, getRandom() will always return 2.

**Constraints:**

- `-2^31 <= val <= 2^31 - 1`
- At most `2 * 10^5` calls will be made to `insert`, `remove`, and `getRandom`.
- There will be **at least one** element in the data structure when `getRandom` is called.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Design: hash map (value → index) + dynamic array, swap-with-last delete

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (list only) | insert/remove O(n), getRandom O(1) | O(n) |
| 1️⃣ Brute force (set only) | insert/remove O(1), getRandom O(n) | O(n) |
| 2️⃣ Optimized (dict + list) | all O(1) average | O(n) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"This is a design question. I need three operations that are all average O(1): insert, remove, and uniform random pick. Let me list what each one needs from the underlying structure."*

| Operation | What it needs |
|---|---|
| `insert` / `remove` | Fast **membership test** ("is `val` already in?") |
| `getRandom` | Fast **random access by position** (pick index `k` uniformly) |

- Duplicates? → No, it's a *set*. Insert of an existing value returns `false`.
- Is `getRandom` ever called on an empty set? → **No**, guaranteed non-empty.
- "Average O(1)" means amortized/expected time is fine (hash tables and dynamic-array appends qualify).
- Order of elements doesn't matter. **This is what lets us swap elements around freely.**

### Step 1 · Brute force

Try each standard structure alone and see where it breaks:

**(a) Python `list` only**
- `getRandom` → `random.choice(list)` → O(1) ✅
- `insert` → must check `val in list` → **O(n)** ❌
- `remove` → `list.remove(val)` searches and then shifts → **O(n)** ❌

**(b) Python `set` only**
- `insert` / `remove` → O(1) ✅
- `getRandom` → a hash set has no "k-th element". You must convert to a list (`random.choice(list(s))`) → **O(n)** ❌

🗣️ *"Each structure alone gets two out of three. The list gives random access, the set gives membership. That suggests combining them, so let me see how to keep them in sync in O(1)."*

In [1]:
import random


class RandomizedSetBrute:
    # list-only version: getRandom O(1), insert/remove O(n)
    def __init__(self):
        self.items = []

    def insert(self, val: int) -> bool:
        if val in self.items:          # O(n) scan
            return False
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.items:      # O(n) scan
            return False
        self.items.remove(val)         # O(n) search + shift
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)

### Step 2 · Optimize

**Plan:** use **both** structures.
- `vals`: a list that holds the elements → O(1) `random.choice`.
- `pos`: a dict `val → index in vals` → O(1) membership and O(1) lookup of *where* a value lives.

`insert(val)`: append to `vals`, record `pos[val] = len(vals) − 1`. Both are O(1).

**The hard part is `remove`.** Deleting from the middle of a list shifts everything after it, which is O(n).

**Key insight: order doesn't matter, so don't delete from the middle. Move the last element into the hole, then pop the end.**
1. `i = pos[val]` (where the victim is)
2. `last = vals[-1]`
3. `vals[i] = last`; `pos[last] = i` (move last into the hole)
4. `vals.pop()`; `del pos[val]` (popping from the end is O(1))

🗣️ *"I'll keep a list for random access and a hash map from value to index for membership. Removing from the middle of a list is O(n) because of shifting, but since order doesn't matter, I overwrite the removed slot with the last element, update that element's index in the map, and pop the tail. Every operation is O(1) on average."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| List only | ❌ | `in` and `remove` are O(n) scans. |
| Set / dict only | ❌ | No indexable positions, so uniform random pick costs O(n). |
| Linked list + dict | ❌ | O(1) delete with a node pointer, but still **no random access by index**. |
| Balanced BST / sorted container | ❌ | O(log n) for everything, and random pick needs order-statistics. That's more complex and slower. |
| Pick random key by retrying random ints | ❌ | Values span ±2³¹, so almost every guess misses. Unbounded time. |
| **dict (val→idx) + list, swap-with-last** | ✅ | Each structure covers the other's weakness, and the swap keeps the list hole-free. |

**Why must the list stay dense (no holes)?** If we left `None` holes, `getRandom` would sometimes land on a hole and have to retry. That breaks the O(1) guarantee as holes pile up.

In [2]:
import random


class RandomizedSet:
    def __init__(self):
        self.vals: list[int] = []        # dense array: O(1) random access
        self.pos: dict[int, int] = {}    # val -> index in vals

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.vals)
        self.vals.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val]
        last = self.vals[-1]
        self.vals[i] = last          # move last element into the hole
        self.pos[last] = i
        self.vals.pop()              # O(1): remove from the end
        del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.vals)

### Step 3 · Toy example walkthrough

Operations: `insert(10)`, `insert(20)`, `insert(30)`, `remove(10)`, `getRandom()`

| op | vals | pos | note |
|---|---|---|---|
| insert(10) | [10] | {10:0} | append |
| insert(20) | [10, 20] | {10:0, 20:1} | append |
| insert(30) | [10, 20, 30] | {10:0, 20:1, 30:2} | append |
| remove(10) step 1–3 | [**30**, 20, 30] | {10:0, 20:1, 30:**0**} | i=0, last=30 copied into hole |
| remove(10) step 4 | [30, 20] | {20:1, 30:0} | pop tail, delete key 10 |
| getRandom() | | | `choice([30, 20])` → each with prob ½ |

**Tricky case: removing the last element itself.** `remove(20)` when `vals = [30, 20]`: i = 1, last = 20. We write `vals[1] = 20` and `pos[20] = 1` (no-op), then pop and `del pos[20]`. It works because we delete `pos[val]` **after** updating `pos[last]`. If we deleted first, we'd re-insert the removed key by mistake.

In [3]:
random.seed(0)

for Impl in (RandomizedSetBrute, RandomizedSet):
    rs = Impl()
    ops = ["insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
    args = [[1], [2], [2], [], [1], [2], []]
    out = []
    for op, a in zip(ops, args):
        out.append(getattr(rs, op)(*a))
    assert out[:3] == [True, False, True]
    assert out[3] in (1, 2)
    assert out[4:] == [True, False, 2]
    print(Impl.__name__, out)

# removing the last element / removing until empty
rs = RandomizedSet()
for v in [5, 6, 7]:
    rs.insert(v)
assert rs.remove(7) and rs.remove(5) and rs.remove(6)
assert rs.vals == [] and rs.pos == {}
assert rs.insert(5)

# fuzz against Python's set semantics, and check list/dict stay in sync
rs, ref = RandomizedSet(), set()
for _ in range(5000):
    v = random.randint(-5, 5)
    if random.random() < 0.5:
        assert rs.insert(v) == (v not in ref); ref.add(v)
    else:
        assert rs.remove(v) == (v in ref); ref.discard(v)
    assert set(rs.vals) == ref and len(rs.vals) == len(ref)
    assert all(rs.vals[i] == x for x, i in rs.pos.items())

# uniformity check
from collections import Counter
rs = RandomizedSet()
for v in range(4):
    rs.insert(v)
freq = Counter(rs.getRandom() for _ in range(40000))
print("getRandom frequencies:", dict(sorted(freq.items())))
assert all(abs(c - 10000) < 600 for c in freq.values())
print("All tests passed ✅")

RandomizedSetBrute [True, False, True, 2, True, False, 2]
RandomizedSet [True, False, True, 1, True, False, 2]
getRandom frequencies: {0: 10001, 1: 10179, 2: 10008, 3: 9812}
All tests passed ✅


### Step 4 · Wrap up

| | insert | remove | getRandom | Space |
|---|---|---|---|---|
| List only | O(n) | O(n) | O(1) | O(n) |
| Set only | O(1) | O(1) | O(n) | O(n) |
| **dict + list** | **O(1)** avg | **O(1)** avg | **O(1)** | **O(n)** |

**Complexity, explained:**
- Dict lookups/inserts/deletes are O(1) **on average** (hashing). The worst case with many collisions is O(n), which is why the problem says "average".
- `list.append` is **amortized** O(1). The list occasionally resizes and copies, but the cost spreads out over many appends.
- `list.pop()` from the end and index assignment are O(1).
- Space: each element is stored twice (once in the list, once as a dict key) → O(n).

**Follow-ups:**
- *Allow duplicates* (LC 381) → `pos` maps `val → set of indices`. Swap-with-last still works, but you update the index sets.
- *Weighted random* → prefix sums plus binary search (LC 528). Not O(1) anymore.
- *Thread safety?* → Mention a lock around each operation.

🗣️ **30-second recap:** *"A list gives O(1) random access but slow membership. A hash map gives O(1) membership but no random access. So I use both: a list of values and a map from value to index. Insert appends. Remove swaps the target with the last element, fixes that element's index, and pops. getRandom is random.choice on the list. Everything is average O(1), with O(n) space."*